In [96]:
import torch
from transformers import AutoModelForCausalLM, AutoProcessor

model_name = "google/gemma-3-4b-it"
device = torch.device("mps")

gemma = AutoModelForCausalLM.from_pretrained(model_name)
processor = AutoProcessor.from_pretrained(model_name)

Loading checkpoint shards: 100%|██████████| 2/2 [00:17<00:00,  8.90s/it]


In [112]:
system_prompt = ""
with open("fact-check-prompt.md") as f:
  system_prompt = f.read()

In [113]:
def generate_gemma_input(current_chat: list, agent_response: str, new_prompt: str) -> list:
  cp = [
    {
      "role": "assistant",
      "content": [{
        "type": "text",
        "text": agent_response,
      }],
    },
    {
      "role": "user",
      "content": [{
        "type": "text",
        "text": new_prompt,
      }],
    },
  ]

  retval = current_chat + cp
  return retval
  

def generate_initial_input(user_prompt: str) -> list:
  return [
    {
      "role": "system",
      "content": [{
        "type": "text",
        "text": system_prompt,
      }]
    },
    {
      "role": "user",
      "content": [{
        "type": "text",
        "text": user_prompt
      }]
    }
  ]

In [114]:
def ask_gemma(user_prompt: str, current_chat: list = None, agent_response: str = None) -> tuple[str, list]:
  if not current_chat:
    prompt = generate_initial_input(user_prompt=user_prompt)
  else:
    prompt = generate_gemma_input(current_chat=current_chat, agent_response=agent_response, new_prompt=user_prompt)

  inputs = processor.apply_chat_template(
    prompt, add_generation_prompt=True, tokenize=True,
    return_dict=True, return_tensors="pt"
  ).to(gemma.device)
  
  input_len = inputs["input_ids"].shape[-1]
  
  with torch.inference_mode():
    output = gemma.generate(**inputs, max_new_tokens=1024, do_sample=False)
    return processor.decode(output[0][input_len:], skip_special_tokens=True), prompt

# print(ask_gemma("Electric vehicles run on nuclear power"))

In [115]:
import re, json
from typing import List
from wikipediaapi import Wikipedia, ExtractFormat
from dataclasses import dataclass

@dataclass(kw_only=True)
class ToolCall:
   name: str | None = None
   parameters: dict | None = None


@dataclass(kw_only=True)
class Evidence:
    source: str
    findings: str

@dataclass(kw_only=True)
class VerificationResult:
    investigation_plan: str
    claim: str
    verdict: str  # TRUE/FALSE/MISLEADING/UNVERIFIED/NEEDS_CONTEXT
    confidence: str  # High/Medium/Low
    evidence: List[Evidence]
    decision_process: str
    explanation: str
    context: str


class Agent:
  def __init__(self):
    self.tools = {}
  
  def register_tool(self, tool_name, tool_method):
    self.tools[tool_name] = tool_method

  def tool_call(self, tool_name, **kwargs):
    print("TOOL CALL:", tool_name, "ARGS:", kwargs)
    if tool_name in self.tools:
      result = self.tools[tool_name](**kwargs)
      print("TOOL RESULT:", result[:100])
      return result
    
  def is_final_result(self, response_text):
    pattern = r'<result>\s*(\{.*?\})\s*</result>'
    
    match = re.search(pattern, response_text, re.DOTALL)
    
    if match:
        json_string = match.group(1)
        try:
            data = json.loads(json_string)
            
            # Parse evidence list
            evidence_list = []
            for evidence_data in data.get('evidence', []):
                evidence_list.append(Evidence(
                    source=evidence_data.get('source', ''),
                    findings=evidence_data.get('findings', '')
                ))
            
            return VerificationResult(
                investigation_plan=data.get('investigation_plan', ''),
                claim=data.get('claim', ''),
                verdict=data.get('verdict', ''),
                confidence=data.get('confidence', ''),
                evidence=evidence_list,
                decision_process=data.get('decision_process', ''),
                explanation=data.get('explanation', ''),
                context=data.get('context', '')
            )
        except (json.JSONDecodeError, KeyError, TypeError):
            return None
    
    return None
  
  def is_thought(self, response_text):
    pattern = r'<think>\s*(\{.*?\})\s*</think>'
    
    match = re.search(pattern, response_text, re.DOTALL)
    
    if match:
        thought = match.group(1)
        return thought
    
    return None
    
  def is_tool_call(self, response_text):
    pattern = r'<tool_call>\s*(\{.*?\})\s*</tool_call>'
    
    match = re.search(pattern, response_text, re.DOTALL)
    
    if match:
        json_string = match.group(1)
        try:
            tool_call_dict = json.loads(json_string)
            return ToolCall(**tool_call_dict)
        except json.JSONDecodeError:
            return None
    
    return None

class EVFactChecker(Agent):
  
  def __init__(self):
    super().__init__()
    self.current_chat = []
    self.wikipedia = Wikipedia(user_agent="EVFactChecker", language="en", extract_format=ExtractFormat.WIKI)

  def _content_downloader(self):
    def inner_downloader(title):
      wiki = self.wikipedia.page(title=title)
      if wiki.exists():
        return wiki.text
    return inner_downloader
  
  def initialize(self):
    self.register_tool("wiki_retrieval", self._content_downloader())

  def process_fact_check(self, prompt):
    (response, chat) = ask_gemma(user_prompt=prompt, current_chat=self.current_chat)
    self.current_chat += chat

    while not (final := self.is_final_result(response)):
        result = response
        if tool_call := self.is_tool_call(response):
          result = self.tool_call(tool_name=tool_call.name, **tool_call.parameters)
        elif thought := self.is_thought(response):
           print("THINK:", thought)
           result = thought
        
        response, _chat = ask_gemma(user_prompt=result,
                                    current_chat=self.current_chat,
                                    agent_response=response)
        
        print("GEMMA:", response)
        print("-----------------------")

        self.current_chat += _chat
    
    return final

In [116]:
from dataclasses import asdict


ev_fact_check = EVFactChecker()
ev_fact_check.initialize()

result = ev_fact_check.process_fact_check("EVs will dominate the market in the near future due to their economical value.")

print(asdict(result))

The following generation flags are not valid and may be ignored: ['top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


TOOL CALL: wiki_retrieval ARGS: {'title': 'Electric vehicle'}


The following generation flags are not valid and may be ignored: ['top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


TOOL RESULT: An electric vehicle (EV) is a motor vehicle whose propulsion is powered fully or mostly by electrici
GEMMA: <result>
{
  "investigation_plan": "I will begin by examining the historical context of electric vehicles, focusing on their early development, decline, and recent resurgence. I will then analyze the provided text for key claims regarding the reasons for the initial decline of EVs and the factors contributing to their recent revival. Finally, I will assess the credibility of the sources cited and provide a clear, objective fact-check with supporting evidence.",
  "claim": "Electric vehicles (EVs) will dominate the market in the near future due to their economical value.",
  "verdict": "MISLEADING",
  "confidence": "Medium",
  "evidence": [
    {
      "source": "https://en.wikipedia.org/wiki/Electric_vehicle",
      "findings": "The text states that EVs have seen a resurgence due to advancements in battery technology, government incentives, and a desire to reduce emi